# Hardware Verification

**Purpose:** This notebook strictly serves to verify your trained PyTorch quantum-classical hybrid model using **real IBM Quantum hardware** for inference. 

**Constraints & Expectations:**
- 🚫 **No Training:** This notebook runs pure inference. No gradients, no optimizers, no weights updates (`eval()` + `torch.no_grad()`).
- ⚡ **Real Hardware:** Uses Qiskit Provider via PennyLane to submit true executions, not simulated runs.
- 🔐 **Configuration:** Requires an `.env` file with `IBM_QUANTUM_TOKEN` alongside the project root.
- 📁 **Cloud/Drive Storage:** Set up to pull your `model.pt` from a configurable path like Google Drive or local storage.

In [ ]:
import os
from pathlib import Path
import torch
!pip install python-dotenv
from dotenv import load_dotenv

# 1. Load Environment Variables (Never hardcode tokens!)
load_dotenv()

IBM_TOKEN = os.getenv("IBM_QUANTUM_TOKEN")
if not IBM_TOKEN:
    raise ValueError("\u274c IBM_QUANTUM_TOKEN not found! Please create a .env file in the root directory with your token.")
else:
    print("\u2705 IBM Quantum Token found.")

# Note: If running on Colab, uncomment the lines below to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Paths & Configuration
Set your model path, testing dataset path, and the number of shots for the quantum device.

In [ ]:
# --- CONFIGURE THESE PATHS ---
# Example for Colab: "/content/drive/MyDrive/models/my_model.pt"
MODEL_PATH = "model.pt"  
# Small subset for quick inference to avoid huge IBM Q queue times
TEST_DATA_DIR = "data/test_samples" 
BATCH_SIZE = 4
SHOTS = 1024

import sys
# Make sure src modules can be found
sys.path.append(os.path.abspath(".")) 

from src.model_builder import build_model
from src.dataset import AudioFeatureDataset
from torch.utils.data import DataLoader
from torchvision import transforms

## 2. IBM Quantum Hardware Setup
We'll query the IBM provider for available compute backends.

In [ ]:
!pip install qiskit-ibm-provider qiskit-ibm-runtime pennylane-qiskit
from qiskit_ibm_provider import IBMProvider
import pennylane as qml

# Authenticate with the provider
try:
    provider = IBMProvider(token=IBM_TOKEN)
except Exception as e:
    # Fallback to save account if running for the first time
    IBMProvider.save_account(token=IBM_TOKEN, overwrite=True)
    provider = IBMProvider()

# List available REAL backends (filter out simulators)
available_backends = [b.name for b in provider.backends() if not b.configuration().simulator]

print("Available Real IBM Quantum Backends:")
for b in available_backends:
    print(f" - {b}")

if not available_backends:
    raise RuntimeError("\u274c No real hardware backends currently available to your account.")

### Select your Backend
Set `BACKEND_NAME` to one of the real hardware instances printed above.

In [ ]:
# Example: 'ibm_kyoto', 'ibm_osaka'
BACKEND_NAME = available_backends[0] if available_backends else None

print(f"Selected Backend: {BACKEND_NAME}")

# Initialize the PennyLane device pointing to the real IBM hardware
# We use 'qiskit.ibmq' device allowing PennyLane to route to IBM Provider
n_qubits = 4 # Adjust based on your model architecture
try:
    dev = qml.device('qiskit.ibmq', wires=n_qubits, backend=BACKEND_NAME, provider=provider, shots=SHOTS)
    print(f"\u2705 PennyLane device connected to {BACKEND_NAME}")
except Exception as e:
    raise RuntimeError(f"\u274c Failed to initialize Qiskit IBMQ device: {e}")

## 3. Model Loading & Data Prep
Ensure the model is loaded as an `eval()` evaluation state.

In [ ]:
device = torch.device("cpu") # Inference on CPU is fine since the QNode handles the heavy quantum lifting

# RECONSTRUCT MODEL SIGNATURE
# You must adjust class_names, base_model depending on your training config
class_names = ['class1', 'class2'] # UPDATE THESE

# Using standard repo build_model logic
# model_config needs to match what was used during training
model_config = {
    'base_model': 'resnet18', 
    'quantum': True,
    'n_qubits': n_qubits,
    'q_depth': 2,
    'max_layers': 10,
    'q_delta': 0.01,
}

print("Instantiating Model...")
model = build_model(model_config, class_names, dataloaders={}, device=device)

# LOAD WEIGHTS
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"\u274c Could not find model at {MODEL_PATH}")

print(f"Loading weights from {MODEL_PATH}...")
state_dict = torch.load(MODEL_PATH, map_location=device)
# Handle potential serialization variations (full model vs state_dict)
if isinstance(state_dict, torch.nn.Module):
    model = state_dict
else:
    model.load_state_dict(state_dict)

model.to(device)
model.eval() # CRITICAL for inference!
print("\u2705 Model loaded and set to evaluation mode.")

# DATA PREPARATION (Minimal batch)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

try:
    test_dataset = AudioFeatureDataset(TEST_DATA_DIR, base_model=model_config['base_model'], transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
except Exception as e:
    print(f"\u26a0\ufe0f Warning: Data load failed. Make sure {TEST_DATA_DIR} exists with valid samples.")
    print(e)
    test_loader = []

## 4. Hardware Inference Execution
We will pass a small batch of inputs to the model. The quantum layers will be offloaded to the IBM cloud queue. This process might take time depending on queue lengths.

In [ ]:
import time
import torch.nn.functional as F

if not test_loader:
    print("No test data found. Creating a random dummy tensor to prove hardware execution...")
    # Base resnet18 expects 3-channel 224x224 images
    dummy_inputs = torch.randn(BATCH_SIZE, 3, 224, 224)
    dummy_labels = torch.randint(0, len(class_names), (BATCH_SIZE,))
    test_loader = [(dummy_inputs, dummy_labels)]

print(f"Executing batch of {BATCH_SIZE} samples on {BACKEND_NAME}...")
start_time = time.time()

# CRITICAL: Pure inference without gradients
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        
        # --- FORWARD PASS ON HARDWARE ---
        logits = model(inputs)
        
        # Metrics
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        print("\n--- Inference Results ---")
        for i in range(len(inputs)):
            print(f"Sample {i+1}:")
            print(f"  Predicted Class: {class_names[preds[i]]}")
            print(f"  Confidence:      {probs[i][preds[i]]:.4f}")
            if labels is not None:
                print(f"  Actual Class:    {class_names[labels[i]]}")
        
        # We only run ONE batch to avoid excessive IBM queueing
        break 

end_time = time.time()
print(f"\n\u2705 Total Hardware Execution Time: {end_time - start_time:.2f} seconds")

## Checklist
- [x] Environment configured (`.env`)
- [x] Correct physical backend selected
- [x] Weights loaded successfully (`eval()`)
- [x] Inference execution finished